## Basic Decoder-Only Transformer Implementation

In [46]:
# Installing files
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-23 17:15:31--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-03-23 17:15:31 (29.3 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [47]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim

In [48]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [49]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [50]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { c:i for i, c in enumerate(chars)}
itos = { i:c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[id] for id in ids])

In [51]:
decode(encode('hello world'))

'hello world'

In [52]:
data = torch.tensor(encode(text), dtype=torch.long)

In [53]:
split = int(0.9 * len(data))
train = data[:split]
valid = data[split:]

In [54]:
torch.manual_seed(100)
block_size = 256
batch_size = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def get_batch(split):
  assert split in ['train', 'valid'], "split is either train or valid"

  if split == "train":
    data = train
  else:
    data = valid

  idxs = data[torch.randint(0, len(data)-block_size, (batch_size, ))]

  x = torch.stack([data[idx:idx+block_size] for idx in idxs])
  y = torch.stack([data[idx+1:idx+block_size+1] for idx in idxs])
  x, y = x.to(device), y.to(device)

  return x, y

In [55]:
xb, yb = get_batch("train")
for i, j in zip(xb[0].tolist(), yb[0].tolist()):
  print(f"{decode([i])} -> {decode([j])}")

e -> a
a -> k
k -> .
. -> 


 -> 


 -> A
A -> l
l -> l
l -> :
: -> 


 -> S
S -> p
p -> e
e -> a
a -> k
k -> ,
, ->  
  -> s
s -> p
p -> e
e -> a
a -> k
k -> .
. -> 


 -> 


 -> F
F -> i
i -> r
r -> s
s -> t
t ->  
  -> C
C -> i
i -> t
t -> i
i -> z
z -> e
e -> n
n -> :
: -> 


 -> Y
Y -> o
o -> u
u ->  
  -> a
a -> r
r -> e
e ->  
  -> a
a -> l
l -> l
l ->  
  -> r
r -> e
e -> s
s -> o
o -> l
l -> v
v -> e
e -> d
d ->  
  -> r
r -> a
a -> t
t -> h
h -> e
e -> r
r ->  
  -> t
t -> o
o ->  
  -> d
d -> i
i -> e
e ->  
  -> t
t -> h
h -> a
a -> n
n ->  
  -> t
t -> o
o ->  
  -> f
f -> a
a -> m
m -> i
i -> s
s -> h
h -> ?
? -> 


 -> 


 -> A
A -> l
l -> l
l -> :
: -> 


 -> R
R -> e
e -> s
s -> o
o -> l
l -> v
v -> e
e -> d
d -> .
. ->  
  -> r
r -> e
e -> s
s -> o
o -> l
l -> v
v -> e
e -> d
d -> .
. -> 


 -> 


 -> F
F -> i
i -> r
r -> s
s -> t
t ->  
  -> C
C -> i
i -> t
t -> i
i -> z
z -> e
e -> n
n -> :
: -> 


 -> F
F -> i
i -> r
r -> s
s -> t
t -> ,
, ->  
  -> y
y -> o
o -> u

In [56]:
dropout = 0.2

class FeedForward(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embd, n_embd * 4),
      nn.ReLU(),
      nn.Linear(n_embd * 4, n_embd),
      nn.Dropout(dropout)
    )

  def forward(self, x):
    return self.net(x)

In [57]:
# self attention intuition
B, T, C = 4, 8, 32
mask = torch.tril(torch.ones((T, T)))
weights = torch.zeros((T, T))
weights = weights.masked_fill(mask==0, float('-inf'))
weights = F.softmax(weights, dim=-1)

x = torch.randn((B, T, C))

result = weights @ x # (B, T, C)

In [58]:
head_size = 32
n_embd = 384
block_size = 256

In [59]:
# self attention
x = torch.randn((B, T, C))

q = nn.Linear(C, head_size)
k = nn.Linear(C, head_size)
v = nn.Linear(C, head_size)

Q = q(x) # (B, T, C)
K = k(x) # (B, T, C)
V = v(x) # (B, T, C)

mask = torch.tril(torch.ones((T, T)))
weights = Q @ K.transpose(-2, -1) # (B, T, T)
weights = weights.masked_fill(mask==0, float('-inf'))
# weights = F.softmax(weights, dim=-1)
# att_vals = weights @ V # (B, T, C)

weights[1]

tensor([[ 2.9144,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.5081, -1.8537,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 5.0308, -3.7545, -2.4110,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-2.6601,  2.6509, -0.5476, -1.4064,    -inf,    -inf,    -inf,    -inf],
        [-1.2823,  0.1181, -1.0440, -1.5066, -1.1325,    -inf,    -inf,    -inf],
        [-0.9810,  0.2144,  0.9340, -2.2084,  0.2094, -0.5683,    -inf,    -inf],
        [-1.1250, -0.4938, -1.5844, -1.7452, -2.1198, -0.4857,  0.2479,    -inf],
        [ 0.5160, -2.7671,  1.1513, -3.8760,  0.0411,  0.4762, -1.7122, -1.9303]],
       grad_fn=<SelectBackward0>)

In [60]:
class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.q = nn.Linear(n_embd, head_size, bias=False)
    self.k = nn.Linear(n_embd, head_size, bias=False)
    self.v = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('mask', torch.tril(torch.ones((block_size, block_size))))
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B, T, C = x.shape

    Q = self.q(x) # (B, T, head_size)
    K = self.k(x) # (B, T, head_size)
    V = self.v(x) # (B, T, head_size)

    weights = (Q @ K.transpose(-2, -1)) * head_size**-0.5 # (B, T, T)
    weights = weights.masked_fill(self.mask[:T, :T]==0, float('-inf'))
    weights = F.softmax(weights, dim=-1)
    weights = self.dropout(weights)
    att_vals = weights @ V # (B, T, head_size)

    return att_vals

In [61]:
class MultiHeadAttention(nn.Module):
  def __init__(self, n_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
    self.proj = nn.Linear(n_heads * head_size, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    x = torch.cat([head(x) for head in self.heads], dim=-1) # (B, T, n_heads * head_size)
    x = self.proj(x) # (B, T, n_embd)
    x = self.dropout(x)
    return x

In [79]:
class Block(nn.Module):
  def __init__(self, n_embd, n_heads):
    super().__init__()
    self.sa = MultiHeadAttention(n_heads, n_embd//n_heads)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    # x -> (B, T, n_embd)
    x = x + self.sa(self.ln1(x)) # (B, T, n_embd)
    x = x + self.ffwd(self.ln2(x)) # (B, T, n_embd)
    return x

In [63]:
class LayerNorm():
  def __init__(self, dim, eps = 1e-5):
    # dim = n
    self.eps = eps
    self.gamma = torch.ones(dim) # (dim)
    self.beta = torch.zeros(dim) # (dim)

  def __call__(self, x):
    # x -> (n_rows, n_cols)
    mean = x.mean(dim=1, keepdim=True) # (n_rows, 1)
    var = x.var(dim=1, keepdim=True) # (n_rows, 1)
    x = (x - mean) / (var + self.eps)**0.5
    x = x * self.gamma + self.beta # ()
    return x


In [75]:
n_blocks = 6
n_heads = 6

class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, n_embd) # (vocab_size, n_embd)
    self.positional = nn.Embedding(block_size, n_embd) # (block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_heads=n_heads) for _ in range(n_blocks)])
    self.ln = nn.LayerNorm(n_embd)
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, x, targets = None):
    # x -> (B, T), targets -> (B, T)
    B, T = x.shape
    token_embd = self.embedding(x) # (B, T, n_embd)
    pos_embd = self.positional(torch.arange(T, device=device)) # (T, n_embd)
    x = token_embd + pos_embd # (B, T, n_embd)
    x = self.blocks(x) # (B, T, n_embd)
    logits = self.lm_head(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      logits = logits.view(B*T, vocab_size) # (B*T, vocab_size)
      targets = targets.view(B*T) # (B*T, )
      loss = F.cross_entropy(logits, targets) # scalar

    return logits, loss

  def generate(self, x, max_new_tokens=10):
    # x -> (B, T)
    for _ in range(max_new_tokens):
      cropped_x = x[:, -block_size:]
      logits, _ = self(cropped_x) # (B, T, vocab_size)
      """
        for each sample in the batch, look at the last character's logits and choose the next character.
        append the character at the end of each sample
      """
      logits = logits[:, -1, :] # (B, vocab_size)
      probs = F.softmax(logits, dim=-1)
      next_token = torch.multinomial(probs, num_samples=1)
      x = torch.cat((x, next_token), dim=-1) # (B, T+1)

    return x

m = BigramLanguageModel()
m = m.to(device)

In [77]:
eval_iters = 200
max_iters = 1000
eval_interval = 50

@torch.no_grad()
def compute_loss():
  # prediction -> (B, T, vocab_size)
  # actual -> (B, T)
  result = {}
  m.eval()

  for split in ['train', 'valid']:
    losses = torch.zeros((eval_iters,))
    for i in range(eval_iters):
      xb, yb = get_batch(split)
      _, loss = m(xb, yb)
      losses[i] = loss.item()
    result[split] = losses.mean()

  m.train()
  return result

In [78]:
# training
batch_size = 32
learning_rate = 3e-4
optimizer = optim.NAdam(m.parameters(), lr=learning_rate)

for i in range(max_iters):
  x, y = get_batch("train")

  # reset gradients
  optimizer.zero_grad(set_to_none=True)

  # forward pass
  logits, loss = m(x, y)

  # backward pass
  loss.backward()

  # update
  optimizer.step()

  # log statistics
  # print(f"Step {i+1} | loss = {loss}")
  if i % eval_interval == 0:
    losses = compute_loss()
    print(f"Step {i}:  Train loss: {losses['train']},  Val loss: {losses['valid']}")

Step 0:  Train loss: 6.030218601226807,  Val loss: 6.797018527984619
Step 50:  Train loss: 0.9095231890678406,  Val loss: 4.932928562164307
Step 100:  Train loss: 0.29768049716949463,  Val loss: 5.674427509307861
Step 150:  Train loss: 0.0913412943482399,  Val loss: 6.560606479644775
Step 200:  Train loss: 0.036618076264858246,  Val loss: 7.082463264465332
Step 250:  Train loss: 0.01742713712155819,  Val loss: 7.534426689147949
Step 300:  Train loss: 0.010424849577248096,  Val loss: 7.871270179748535
Step 350:  Train loss: 0.007298960816115141,  Val loss: 8.288742065429688
Step 400:  Train loss: 0.0067977518774569035,  Val loss: 8.529260635375977
Step 450:  Train loss: 1.7974971532821655,  Val loss: 6.432097911834717
Step 500:  Train loss: 0.3048589527606964,  Val loss: 5.70404052734375
Step 550:  Train loss: 0.1518433541059494,  Val loss: 6.121441841125488
Step 600:  Train loss: 0.0870552659034729,  Val loss: 6.427028179168701
Step 650:  Train loss: 0.06158382073044777,  Val loss: 6.8

In [ ]:
# Loss logs
# Single head attention: 2.1058027744293213
# Multi head attention: 1.5715726613998413

In [80]:
# generate output
xt = torch.tensor([encode("a")]).to(device)
yt = m.generate(xt, 1000)

outputs = [decode(output.tolist()) for output in yt]
for output in outputs:
  print(output)

ak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We.
We know't, we wet.
First Cit.

Letize us kill hit him, we't us wend we't knd wetim, him, we we us him, pet we we hile'thime we plllllllle't.
We we we't wet.
Fillll w't s ust knd Cizet we usit.


Fitit wend us us kimil himy him, kimitizend hiend plle we hilenowend.
We we st.
We s we w't hillllll us knd knowe we't us knd h?




Wet us hit hire kill:
We we hit we knd we us we't hire't.

Le him, we we Allll we hit lle't, anowe we we killll hire't.

Wet we spl:
Wen:
Fil we ust w't Cirs us kit and kit l:
Lend knd him, us him, himim, we we't.

Allllllll:
Wet knd wet kim, kim, we kit hilll s we s s Cil st spl wen:
Wet uspllllll k, us hit.

Le w't hirs us knowetil himit hizenowe't w't us pend.
Le wenowethize't we himill Cim, hirenowendit, s spll splllle dit usple t wet.
Wetowe hilllet.
We us and